# 编程练习：Linear Regression

欢迎来到第一个编程练习！在本实验中，你将实现单变量 linear regression（线性回归），预测连锁餐厅的利润。

## 目录

- [1 - 导入依赖库](#1)
- [2 - 问题描述](#2)
- [3 - 数据集](#3)
- [4 - 回顾 Linear Regression](#4)
- [5 - 计算 Cost](#5)
  - [练习 1](#ex01)
- [6 - Gradient Descent](#6)
  - [练习 2](#ex02)
- [7 - 使用 Batch Gradient Descent 学习参数](#7)


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除本 Notebook 中不参与评分的单元，也不要添加新单元。
>
> **通过本作业后**，如果想尝试修改不参与评分的代码，可以按照 Notebook 底部的说明操作。


<a name="1"></a>
## 1 - 导入依赖库

首先运行下面的代码单元，导入本作业所需的库。

- [NumPy](https://www.numpy.org) 是 Python 中用于处理 matrix（矩阵）等数值数据的基础库。
- [Matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- `utils.py` 包含本作业使用的辅助函数，无需修改该文件中的代码。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
import copy
import math
%matplotlib inline

<a name="2"></a>
<a name="2.1"></a>
## 2 - 问题描述

假设你是一家连锁餐厅的 CEO，正在考虑到哪些城市开设新店。

- 你希望将业务扩展到可能带来更高利润的城市。
- 连锁餐厅已经在多个城市开店，你掌握了这些城市的人口数据和餐厅利润数据。
- 对于准备开设新店的候选城市，你也掌握了它们的人口数据。

你能否利用这些数据，找出可能为餐厅带来更高利润的城市？

<a name="3"></a>
<a name="2.2"></a>
## 3 - 数据集

首先加载本任务的数据集。下面的 `load_data()` 函数将数据加载到 `x_train` 和 `y_train` 中：

- `x_train` 表示城市人口。
- `y_train` 表示餐厅在对应城市的利润，负值表示亏损。
- `x_train` 和 `y_train` 都是 NumPy array（数组）。


In [ ]:
# load the dataset
x_train, y_train = load_data()

### 查看变量

开始任务之前，先熟悉数据集会很有帮助。可以从打印各个变量、查看其中的内容开始。

下面的代码打印变量 `x_train` 及其类型。


In [ ]:
# print x_train
print("Type of x_train:",type(x_train))
print("First five elements of x_train are:\n", x_train[:5]) 

`x_train` 是一个 NumPy array，其中的小数都大于 0。

- 这些值以 10,000 人为单位表示城市人口。
- 例如，6.1101 表示该城市有 61,101 人。

下面打印 `y_train`。


In [ ]:
# print y_train
print("Type of y_train:",type(y_train))
print("First five elements of y_train are:\n", y_train[:5])  

类似地，`y_train` 也是一个 NumPy array，其中的小数既有正值，也有负值。

- 这些值表示餐厅在各城市的平均月利润，单位为 10,000 美元。
- 例如，17.592 表示该城市餐厅的平均月利润为 175,920 美元。
- -2.6807 表示平均月利润为 -26,807 美元，也就是每月平均亏损 26,807 美元。


### 检查变量的维度

查看数据的维度，也是熟悉数据集的一种有效方式。

请打印 `x_train` 和 `y_train` 的 shape，确认数据集中有多少个训练样本。


In [ ]:
print ('The shape of x_train is:', x_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(x_train))

城市人口 array 包含 97 个数据点，平均月利润 array 也包含 97 个数据点。它们都是 NumPy 1-D array。


### 可视化数据

通过可视化可以更直观地理解数据。

- 这个数据集只有两个需要绘制的属性（利润和人口），因此可以用 scatter plot（散点图）展示。
- 实际问题往往包含两个以上的属性，例如人口、家庭平均收入、月利润和月销售额。即使如此，仍然可以用 scatter plot 分别观察每一对属性之间的关系。


In [ ]:
# Create a scatter plot of the data. To change the markers to red "x",
# we used the 'marker' and 'c' parameters
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')
plt.show()

你的目标是构建一个 linear regression 模型来拟合这些数据。

训练完成后，输入一个新城市的人口，模型就可以估计餐厅在该城市可能获得的月利润。


<a name="4"></a>
<a name="2.3"></a>
## 4 - 回顾 Linear Regression

在本练习中，你将根据数据集拟合 linear regression 的参数 $(w,b)$。模型将 `x`（城市人口）映射为 `y`（餐厅在该城市的月利润）：

$$
f_{w,b}(x)=wx+b
$$

训练模型的目标是找到最适合数据集的参数 $(w,b)$。可以使用 cost function（代价函数）$J(w,b)$ 比较不同参数的表现：

- $J$ 是关于 $(w,b)$ 的函数，其值取决于参数 $(w,b)$。
- 最适合数据的参数组合，会使 $J(w,b)$ 最小。

为了找到使 $J(w,b)$ 尽可能小的参数，可以使用 **gradient descent（梯度下降）**。在合适的 learning rate 下，每次更新都使参数逐步接近能获得最低 cost 的最优值。

训练后的模型接收输入 feature $x$（城市人口），输出预测值 $f_{w,b}(x)$（餐厅在该城市的预计月利润）。


<a name="5"></a>
<a name="2.4"></a>
## 5 - 计算 Cost

Gradient descent 反复调整参数 $(w,b)$，使 cost $J(w,b)$ 逐渐减小。每次参数更新后，计算 $J(w,b)$ 有助于观察训练进展。本节将实现计算 $J(w,b)$ 的函数，以便检查 gradient descent 的运行情况。

### Cost Function

回顾课程内容，单变量 linear regression 的 cost function 定义为：

$$
J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2
$$

- $f_{w,b}(x^{(i)})$ 是模型预测的餐厅利润，$y^{(i)}$ 是数据中记录的实际利润。
- $m$ 是数据集中的训练样本数。

### 模型预测

对于单变量 linear regression，模型对样本 $x^{(i)}$ 的预测为：

$$
f_{w,b}(x^{(i)})=wx^{(i)}+b
$$

这是一个直线方程，其中 intercept（截距）为 $b$，slope（斜率）为 $w$。

### 实现

请完成下面的 `compute_cost()` 函数，计算 cost $J(w,b)$。


<a name="ex01"></a>
### 练习 1

请完成 `compute_cost` 函数，实现以下步骤：

1. 遍历所有训练样本，计算每个样本的预测值：

   $$
   f_{w,b}(x^{(i)})=wx^{(i)}+b
   $$

2. 计算每个样本的 squared error（平方误差）：

   $$
   cost^{(i)}=\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2
   $$

3. 对所有样本的 squared error 求和，并除以 $2m$，返回总 cost：

   $$
   J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}cost^{(i)}
   $$

其中，$m$ 是训练样本数，$\sum$ 是求和符号。

如果遇到困难，可以查看下面代码单元之后的提示。


In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_cost

def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
        x (ndarray): Shape (m,) Input to the model (Population of cities) 
        y (ndarray): Shape (m,) Label (Actual profits for the cities)
        w, b (scalar): Parameters of the model
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    # You need to return this variable correctly
    total_cost = 0
    
    ### START CODE HERE ###
    
    ### END CODE HERE ### 

    return total_cost

<details>
<summary><span style="color:darkgreen;"><strong>点击查看提示</strong></span></summary>

求和表达式（例如 $h=\sum_{i=0}^{m-1}2i$）可以用以下代码实现：

```python
h = 0
for i in range(m):
    h = h + 2*i
```

在本题中，可以使用 `for` loop 遍历 `x` 中的所有样本，将每次迭代得到的 `cost` 累加到循环外初始化的变量 `cost_sum` 中。

最后，用 `cost_sum` 除以 `2 * m`，得到并返回 `total_cost`。

如果你刚开始学习 Python，请检查代码缩进是否一致。错误的缩进可能导致结果不正确，或触发 `IndentationError: unexpected indent`。详情可参阅社区中的[相关讨论](https://community.deeplearning.ai/t/indentation-in-python-indentationerror-unexpected-indent/159398)。

<details>
<summary><span style="color:darkblue;"><strong>点击查看更多提示</strong></span></summary>

该函数的整体结构可以写成：

```python
def compute_cost(x, y, w, b):
    # number of training examples
    m = x.shape[0] 

    # You need to return this variable correctly
    total_cost = 0

    ### START CODE HERE ###  
    # Variable to keep track of sum of cost from each example
    cost_sum = 0

    # Loop over training examples
    for i in range(m):
        # Your code here to get the prediction f_wb for the ith example
        f_wb = 
        # Your code here to get the cost associated with the ith example
        cost = 

        # Add to sum of cost for each example
        cost_sum = cost_sum + cost 

    # Get the total cost as the sum divided by (2*m)
    total_cost = (1 / (2 * m)) * cost_sum
    ### END CODE HERE ### 

    return total_cost
```

如果仍有困难，可以展开下面的提示，查看如何计算 `f_wb` 和 `cost`。

<details>
<summary><span style="color:darkblue;"><strong>计算 f_wb 的提示</strong></span></summary>

对于 scalar（标量）$a$、$b$ 和 $c$，表达式 $h=ab+c$ 可以写成 `h = a * b + c`。这里的 `x[i]`、`w` 和 `b` 都是 scalar。

<details>
<summary><span style="color:blue;"><strong>计算 f_wb 的更多提示</strong></span></summary>

可以使用 `f_wb = w * x[i] + b` 计算预测值。

</details>

</details>

<details>
<summary><span style="color:darkblue;"><strong>计算 cost 的提示</strong></span></summary>

变量 `z` 的平方可以写成 `z**2`。

<details>
<summary><span style="color:blue;"><strong>计算 cost 的更多提示</strong></span></summary>

可以使用 `cost = (f_wb - y[i]) ** 2` 计算该样本的 cost。

</details>

</details>

</details>

</details>


运行下面的测试代码，检查你的实现是否正确：


In [ ]:
# Compute cost with some initial values for paramaters w, b
initial_w = 2
initial_b = 1

cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')

# Public tests
from public_tests import *
compute_cost_test(compute_cost)

**预期输出：** 初始 `w` 对应的 cost 为 `75.203`。


<a name="6"></a>
<a name="2.5"></a>
## 6 - Gradient Descent

本节将实现 linear regression 中参数 $w$ 和 $b$ 对应的 gradient 计算。


如课程视频所述，gradient descent 重复执行以下更新，直到收敛：

$$
\begin{aligned}
b &:= b-\alpha\frac{\partial J(w,b)}{\partial b} \\
w &:= w-\alpha\frac{\partial J(w,b)}{\partial w}
\end{aligned}
\tag{1}
$$

参数 $w$ 和 $b$ 必须**同时更新**，对应的 partial derivative（偏导数）为：

$$
\frac{\partial J(w,b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right) \tag{2}
$$

$$
\frac{\partial J(w,b)}{\partial w}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)x^{(i)} \tag{3}
$$

- $m$ 是数据集中的训练样本数。
- $f_{w,b}(x^{(i)})$ 是模型的预测值，$y^{(i)}$ 是 target（目标值）。

你将实现 `compute_gradient` 函数，计算 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$。


<a name="ex02"></a>
### 练习 2

请完成 `compute_gradient` 函数，实现以下步骤。

首先遍历所有训练样本，计算每个样本的模型预测值：

$$
f_{w,b}(x^{(i)})=wx^{(i)}+b
$$

然后计算该样本对参数 $w$ 和 $b$ 的 gradient 的贡献：

$$
\left(\frac{\partial J(w,b)}{\partial b}\right)^{(i)}=f_{w,b}(x^{(i)})-y^{(i)}
$$

$$
\left(\frac{\partial J(w,b)}{\partial w}\right)^{(i)}=\left(f_{w,b}(x^{(i)})-y^{(i)}\right)x^{(i)}
$$

最后，对所有样本的贡献求和并取平均，返回完整的 gradient：

$$
\frac{\partial J(w,b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}\left(\frac{\partial J(w,b)}{\partial b}\right)^{(i)}
$$

$$
\frac{\partial J(w,b)}{\partial w}=\frac{1}{m}\sum_{i=0}^{m-1}\left(\frac{\partial J(w,b)}{\partial w}\right)^{(i)}
$$

其中，$m$ 是训练样本数，$\sum$ 是求和符号。

如果遇到困难，可以查看下面代码单元之后的提示。


In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_gradient
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities) 
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    
    # Number of training examples
    m = x.shape[0]
    
    # You need to return the following variables correctly
    dj_dw = 0
    dj_db = 0
    
    ### START CODE HERE ###
    
    ### END CODE HERE ### 
        
    return dj_dw, dj_db

<details>
<summary><span style="color:darkgreen;"><strong>点击查看提示</strong></span></summary>

求和表达式（例如 $h=\sum_{i=0}^{m-1}2i$）可以用以下代码实现：

```python
h = 0
for i in range(m):
    h = h + 2*i
```

在本题中，可以使用 `for` loop 遍历 `x` 中的所有样本，将每个样本对应的 gradient 分别累加到循环外初始化的变量 `dj_dw` 和 `dj_db` 中。

最后，将 `dj_dw` 和 `dj_db` 分别除以 `m` 并返回。

<details>
<summary><span style="color:darkblue;"><strong>点击查看更多提示</strong></span></summary>

该函数的整体结构可以写成：

```python
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities) 
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
    """

    # Number of training examples
    m = x.shape[0]

    # You need to return the following variables correctly
    dj_dw = 0
    dj_db = 0

    ### START CODE HERE ### 
    # Loop over examples
    for i in range(m):  
        # Your code here to get prediction f_wb for the ith example
        f_wb = 

        # Your code here to get the gradient for w from the ith example 
        dj_dw_i = 

        # Your code here to get the gradient for b from the ith example 
        dj_db_i = 

        # Update dj_db : In Python, a += 1  is the same as a = a + 1
        dj_db += dj_db_i

        # Update dj_dw
        dj_dw += dj_dw_i

    # Divide both dj_dw and dj_db by m
    dj_dw = dj_dw / m
    dj_db = dj_db / m
    ### END CODE HERE ### 

    return dj_dw, dj_db
```

如果仍有困难，可以展开下面的提示，查看如何计算 `f_wb`、`dj_dw_i` 和 `dj_db_i`。

<details>
<summary><span style="color:darkblue;"><strong>计算 f_wb 的提示</strong></span></summary>

你已在上一个练习中完成过这个计算！对于 scalar $a$、$b$ 和 $c$，表达式 $h=ab+c$ 可以写成 `h = a * b + c`。这里的 `x[i]`、`w` 和 `b` 都是 scalar。

<details>
<summary><span style="color:blue;"><strong>计算 f_wb 的更多提示</strong></span></summary>

可以使用 `f_wb = w * x[i] + b` 计算预测值。

</details>

</details>

<details>
<summary><span style="color:darkblue;"><strong>计算 dj_dw_i 的提示</strong></span></summary>

对于 scalar $a$、$b$ 和 $c$，表达式 $h=(a-b)c$ 可以写成 `h = (a-b)*c`。这里的 `f_wb`、`y[i]` 和 `x[i]` 都是 scalar。

<details>
<summary><span style="color:blue;"><strong>计算 dj_dw_i 的更多提示</strong></span></summary>

可以使用 `dj_dw_i = (f_wb - y[i]) * x[i]` 计算该样本对 `w` 的 gradient 的贡献。

</details>

</details>

<details>
<summary><span style="color:darkblue;"><strong>计算 dj_db_i 的提示</strong></span></summary>

可以使用 `dj_db_i = f_wb - y[i]` 计算该样本对 `b` 的 gradient 的贡献。

</details>

</details>

</details>


运行下面的代码单元，分别使用两组初始参数 $w$ 和 $b$，检查 `compute_gradient` 函数的实现。


In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

compute_gradient_test(compute_gradient)

下面是初始参数均为 0 时的 gradient 检查结果。

**预期输出：**

| 参数设置 | `dj_dw` | `dj_db` |
| --- | --- | --- |
| 初始 `w = 0, b = 0` | -65.32884975 | -5.83913505154639 |


In [ ]:
# Compute and display cost and gradient with non-zero w
test_w = 0.2
test_b = 0.2
tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, test_w, test_b)

print('Gradient at test w, b:', tmp_dj_dw, tmp_dj_db)

**预期输出：**

| 参数设置 | `dj_dw` | `dj_db` |
| --- | --- | --- |
| 测试用 `w, b` | -47.41610118 | -4.007175051546391 |


<a name="7"></a>
<a name="2.6"></a>
## 7 - 使用 Batch Gradient Descent 学习参数

现在使用 batch gradient descent（批量梯度下降）寻找 linear regression 模型的最优参数。这里的 batch 表示每次迭代都使用全部训练样本。

- 本部分无需编写代码，直接运行下面的代码单元即可。
- 检查 gradient descent 是否正常工作的一种方法，是观察 $J(w,b)$ 是否随每次更新而下降。
- 如果 gradient 和 cost 的计算正确，并且 learning rate $\alpha$ 设置合适，$J(w,b)$ 应持续下降或保持不变，并在算法结束时接近一个稳定值。


In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(x)
    
    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(x, y, w, b )  

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w, b, J_history, w_history #return w and J,w history for graphing

下面运行上述 gradient descent 算法，学习本数据集对应的模型参数。


In [ ]:
# initialize fitting parameters. Recall that the shape of w is (n,)
initial_w = 0.
initial_b = 0.

# some gradient descent settings
iterations = 1500
alpha = 0.01

w,b,_,_ = gradient_descent(x_train ,y_train, initial_w, initial_b, 
                     compute_cost, compute_gradient, alpha, iterations)
print("w,b found by gradient descent:", w, b)

**预期输出：**

| Gradient descent 得到的 `w` | Gradient descent 得到的 `b` |
| --- | --- |
| 1.16636235 | -3.63029143940436 |


下面使用 gradient descent 得到的最终参数，绘制 linear fit（线性拟合）结果。

回顾单个样本的预测公式：$f(x^{(i)})=wx^{(i)}+b$。

要计算整个数据集的预测值，可以遍历所有训练样本，逐一计算。下面的代码展示了这一过程。


In [ ]:
m = x_train.shape[0]
predicted = np.zeros(m)

for i in range(m):
    predicted[i] = w * x_train[i] + b

下面绘制预测值，查看 linear fit 的效果。


In [ ]:
# Plot the linear fit
plt.plot(x_train, predicted, c = "b")

# Create a scatter plot of the data. 
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')

最终参数 $w,b$ 也可以用来预测新城市的利润。下面分别预测人口为 35,000 和 70,000 的地区可能带来的利润。

- 模型输入的人口以 10,000 人为单位。
- 因此，35,000 人对应的模型输入为 `np.array([3.5])`。
- 类似地，70,000 人对应的模型输入为 `np.array([7.])`。


In [ ]:
predict1 = 3.5 * w + b
print('For population = 35,000, we predict a profit of $%.2f' % (predict1*10000))

predict2 = 7.0 * w + b
print('For population = 70,000, we predict a profit of $%.2f' % (predict2*10000))

**预期输出：**

| 城市人口 | 预测月利润（美元） |
| --- | --- |
| 35,000 | 4519.77 |
| 70,000 | 45342.45 |


**恭喜完成 Linear Regression 编程练习！下周，你将构建模型来解决另一类问题：classification（分类）。下周见！**


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="../../../assets/images/unlock_cells.gif" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
